In [1]:
from urllib.request import urlopen
from bs4 import BeautifulSoup as bs
import pandas as pd

In [2]:
url = "https://www.globalmilitary.net/?sort=nuclear_total"
client = urlopen(url)
html = client.read()

In [3]:
html

b'\n\n\n\n<!DOCTYPE html>\n\n<html lang="en" dir="ltr">\n<head>\n\n  <title>\n  \n  \n    GlobalMilitary.net - World Military Forces Database 2026\n  \n</title>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  \n  <!-- DNS prefetch and preconnect for faster loading -->\n  <link rel="dns-prefetch" href="https://cdnjs.cloudflare.com">\n  <link rel="dns-prefetch" href="https://www.googletagmanager.com">\n  <link rel="dns-prefetch" href="https://pagead2.googlesyndication.com">\n  <!-- CSS loading -->\n  <link rel="stylesheet" href="https://media.globalmilitary.net/world/css/style.css">\n  \n  <!-- Critical theme initialization - prevent FOUC -->\n  <script>\n  (function() {\n    // Apply stored theme immediately to prevent flash\n    var theme = localStorage.getItem(\'theme\') || \n                (document.cookie.match(/theme=([^;]+)/) ? \n                 document.cookie.match(/theme=([^;]+)/)[1] : \'light\');\n    if (theme === \'dar

In [4]:
client.close()

In [5]:
soup = bs(html, "html.parser")
soup


<!DOCTYPE html>

<html dir="ltr" lang="en">
<head>
<title>
  
  
    GlobalMilitary.net - World Military Forces Database 2026
  
</title>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<!-- DNS prefetch and preconnect for faster loading -->
<link href="https://cdnjs.cloudflare.com" rel="dns-prefetch"/>
<link href="https://www.googletagmanager.com" rel="dns-prefetch"/>
<link href="https://pagead2.googlesyndication.com" rel="dns-prefetch"/>
<!-- CSS loading -->
<link href="https://media.globalmilitary.net/world/css/style.css" rel="stylesheet"/>
<!-- Critical theme initialization - prevent FOUC -->
<script>
  (function() {
    // Apply stored theme immediately to prevent flash
    var theme = localStorage.getItem('theme') || 
                (document.cookie.match(/theme=([^;]+)/) ? 
                 document.cookie.match(/theme=([^;]+)/)[1] : 'light');
    if (theme === 'dark') {
      document.body.classList.add('dark');
    }
  })();
  

In [6]:
continer = soup.find_all("div", {"class": "table-container table-responsive"})
continer

[<div class="table-container table-responsive">
 <table class="table">
 <thead>
 <tr>
 <th class="orderable">
 <a href="?sort=name">Country</a>
 </th>
 <th class="header orderable">
 <a href="?sort=military_power_index">Military Index</a>
 </th>
 <th class="header orderable">
 <a href="?sort=milex_usd">Military Budget</a>
 </th>
 <th class="header orderable">
 <a href="?sort=troops_active">Active Troops</a>
 </th>
 <th class="header orderable">
 <a href="?sort=aircrafts">Aircraft</a>
 </th>
 <th class="header orderable">
 <a href="?sort=ships">War Ships</a>
 </th>
 <th class="asc header orderable">
 <a href="?sort=-nuclear_total">Nuclear Warheads</a>
 </th>
 </tr>
 </thead>
 <tbody>
 <tr class="even">
 <td>🇷🇺 <a href="/countries/rus/">Russia</a></td>
 <td><span class="power-index-badge">82.2</span></td>
 <td>$149.0B</td>
 <td class="">1100k</td>
 <td class=""><a href="/air_forces/rus/">4300</a></td>
 <td class=""><a href="/navies/rus/">565</a></td>
 <td class=""><a href="/nuclear/rus/"

In [7]:
len(continer)

11

In [8]:
import re

soup = bs(html, "html.parser")

# Find all tables
tables = soup.find_all("table", {"class": "table"})

# Extract data from tables
all_data = []

for table in tables:
    rows = table.find_all("tr")
    for row in rows[1:]:  # Skip header row
        cols = row.find_all("td")
        if cols:
            data = [col.get_text(strip=True) for col in cols]
            # Clean country name - remove emoji only
            if data:
                country = data[0]
                # Remove flag emoji and space, keep country name
                country_name = re.sub(r'^[\U0001F1E6-\U0001F1FF]{2}\s*', '', country)
                data[0] = country_name
            all_data.append(data)

# Create DataFrame
df = pd.DataFrame(all_data, columns=["Country", "Military Index", "Military Budget", "Active Troops", "Aircraft", "War Ships", "Nuclear Warheads"])

# Display all rows
print(df)

                 Country Military Index Military Budget Active Troops  \
0                 Russia           82.2         $149.0B         1100k   
1          United States           85.2         $997.3B         1326k   
2                  China           81.4         $313.7B         2035k   
3                 France           64.2          $64.7B          270k   
4         United Kingdom           61.9          $81.8B          144k   
..                   ...            ...             ...           ...   
157             Eswatini            8.4           $0.1B             —   
158               Gambia            8.1           $0.0B            1k   
159  Antigua and Barbuda            7.4           $0.0B            0k   
160             Maldives            4.7           $0.0B             —   
161               Bhutan            3.8           $0.0B             —   

    Aircraft War Ships Nuclear Warheads  
0       4300       565             5459  
1      12860       474             5042

In [9]:
# Clean data - remove special characters
df = df.apply(lambda x: x.str.replace(r'[^\w\s\-\.]', '', regex=True) if x.dtype == 'object' else x)

# Remove extra whitespace
df = df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

# Display as formatted table
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(df.to_string(index=False))

                  Country Military Index Military Budget Active Troops Aircraft War Ships Nuclear Warheads
                   Russia           82.2         $149.0B         1100k     4300       565             5459
            United States           85.2         $997.3B         1326k    12860       474             5042
                    China           81.4         $313.7B         2035k     3440      1036              600
                   France           64.2          $64.7B          270k      988       138              370
           United Kingdom           61.9          $81.8B          144k      640        80              225
                    India           74.4          $86.1B         1476k     2217       291              180
                 Pakistan           67.9          $10.2B          660k     1369       148              170
                   Israel           63.6          $46.5B          170k      532        53               90
              South Korea           6

In [10]:
df = df.replace('—', 0)

In [11]:
def clean_military_data(value):
    if isinstance(value, str):
        value = value.replace('$', '').replace(',', '').strip()
        if 'B' in value:
            return float(value.replace('B', '')) * 1_000_000_000 
        if 'M' in value:
            return float(value.replace('M', '')) * 1_000_000 
        if 'k' in value:
            return float(value.replace('k', '')) * 1_000
        
        try:
            return float(value)
        except ValueError:
            return 0.0
    return value

df['Military Budget'] = df['Military Budget'].apply(clean_military_data)
df['Active Troops'] = df['Active Troops'].apply(clean_military_data)
 
df.to_csv('military_data_tableau_ready.csv', index=False)

print(df.head())

          Country Military Index  Military Budget  Active Troops Aircraft  \
0          Russia           82.2     1.490000e+11      1100000.0     4300   
1   United States           85.2     9.973000e+11      1326000.0    12860   
2           China           81.4     3.137000e+11      2035000.0     3440   
3          France           64.2     6.470000e+10       270000.0      988   
4  United Kingdom           61.9     8.180000e+10       144000.0      640   

  War Ships Nuclear Warheads  
0       565             5459  
1       474             5042  
2      1036              600  
3       138              370  
4        80              225  


In [12]:
df.head()

,Country,Military Index,Military Budget,Active Troops,Aircraft,War Ships,Nuclear Warheads
0,Russia,82.2,1.490000e+11,1100000.0,4300,565,5459
1,United States,85.2,9.973000e+11,1326000.0,12860,474,5042
2,China,81.4,3.137000e+11,2035000.0,3440,1036,600
3,France,64.2,6.470000e+10,270000.0,988,138,370
4,United Kingdom,61.9,8.180000e+10,144000.0,640,80,225


In [13]:
df.to_csv("military_data_final_done.csv", index=False)